# The simulator and judge prompts

Every LLM prompt the harness sends, printed in the case notebook's layout
(`01_cases`) so they fit a page. The text is the file under `src/prompts/` with
its markdown stripped: a `##` heading becomes the left-hand label, a bullet
becomes `·`, and `{{placeholders}}` are kept as they are, because they are
where the case, the reply or the rubric is filled in.

The header names the version that is logged with every call (the file's
`<!-- version -->` comment, not its file name, so `judge_pointwise_v1.md`
reads `judge-pointwise/2`), what calls the prompt, at what temperature, and
the user message that comes with it as the system prompt. `judge_pairwise_v1`
is left out: `judge_pairwise_v2` replaced it, and no stored comparison uses it.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))  # the subtree, so `src` imports

import re, textwrap
from src.llm import PROMPTS_DIR, load_prompt

WIDTH = 90

# What calls each prompt, at what temperature, and with which user message
# (src/simulator.py, src/judge.py, src/participant_judge.py).
PROMPTS = {
    "simulator_v1": ("simulator, member turns 2-3", "temperature 0.7",
                     "the conversation so far, as You: / Tonic: lines"),
    "judge_pointwise_v1": ("point-wise judge, simulated executions", "temperature 0",
                           "'Score the reply.'"),
    "judge_pairwise_v2": ("pair-wise judge, follow-up vs. baseline, both orderings", "temperature 0",
                          "'Compare the replies.'"),
    "judge_participant_v1": ("participant judge, opening turn of each request", "temperature 0",
                             "'Score the reply.'"),
    "participant_rubrics_v1": ("the criteria judge_participant_v1 is filled with", "one set per request type"),
}

# A heading becomes a short label; anything not listed falls back to the heading.
LABELS = {
    "Who you are": "persona", "What you want from this conversation": "goal",
    "Rules": "rules", "Output": "output", "The member's request": "request",
    "The assistant's reply": "reply", "Reply A": "reply A", "Reply B": "reply B",
    "When each reply was given": "dates", "What the assistant did": "did",
    "Criteria": "criteria", "Reference material": "references",
    "How to decide": "decide", "Scale": "scale",
}


PLACEHOLDER = re.compile(r"^\{\{\w+\}\}$")


def label_for(heading, parent=None):
    key = heading.split(" (")[0]
    if parent == "dates":  # the two `### Reply A/B` under the dates section
        return "date " + key[-1]
    return LABELS.get(key, key)


def sections(body):
    """(label, lines) pairs: the preamble first, then one per heading. Prose
    after a `###` sub-section's placeholder belongs to the parent section, so
    it is printed unlabelled rather than under the sub-section."""
    out, label, lines, parent, sub = [], "prompt", [], None, False
    for line in body.splitlines():
        m = re.match(r"^(#{2,3}) (.+)$", line)
        if m:
            out.append((label, lines))
            sub = len(m.group(1)) == 3
            label = label_for(m.group(2), parent) if sub else label_for(m.group(2))
            parent = parent if sub else label
            lines = []
        elif sub and line.strip() and not PLACEHOLDER.match(line) and any(x.strip() for x in lines):
            out.append((label, lines))
            label, lines, sub = "", [line], False
        else:
            lines.append(line)
    out.append((label, lines))
    return [(l, ls) for l, ls in out if any(x.strip() for x in ls)]


def blocks(lines):
    """Paragraphs and bullets, each joined onto one line."""
    out, cur = [], None
    for raw in lines:
        line = raw.strip()
        if not line:
            cur = None
        elif PLACEHOLDER.match(line):
            out.append(["", line])
            cur = None
        elif line.startswith("- "):
            cur = ["· ", line[2:]]
            out.append(cur)
        elif cur is not None and (raw.startswith("  ") or cur[0] == ""):
            cur[1] += " " + line
        else:
            cur = ["", line]
            out.append(cur)
    return [(b, re.sub(r"`|\*\*", "", t)) for b, t in out]


def card(name):
    prompt = load_prompt(name)
    parts = sections(prompt.template)
    pad = max(10, max(len(l) for l, _ in parts) + 2)
    used, *rest = PROMPTS[name]
    print("═" * 100)
    print(f"{name}.md   [{' · '.join([prompt.version] + rest[:1])}]")
    print("used by:".ljust(pad) + used)
    if rest[1:]:
        print("user msg:".ljust(pad) + rest[1])
    for label, lines in parts:
        first = True
        for bullet, text in blocks(lines):
            head = (label + ":").ljust(pad) if first and label else " " * pad
            print(textwrap.fill(text, WIDTH, initial_indent=head + bullet,
                                subsequent_indent=" " * (pad + len(bullet))))
            first = False

In [2]:
for name in PROMPTS:
    card(name)

════════════════════════════════════════════════════════════════════════════════════════════════════
simulator_v1.md   [simulator/1 · temperature 0.7]
used by:  simulator, member turns 2-3
user msg: the conversation so far, as You: / Tonic: lines
prompt:   You are playing one member of a small group planning a trip together in the Trip
          Tonic app. You are talking to Tonic, the app's planning assistant, in the trip's
          chat. Stay in character and pursue the goal below the way this person would:
          their words, their level of detail, their patience.
persona:  {{user_profile}}
goal:     {{goal}}
rules:    · Write only what this member would type into the chat: one message, natural,
            short. No stage directions, no commentary about the assistant, no lists of
            everything at once — say the next thing.
          · Never start a message with a slash command. Never mention being simulated.
          · When the goal has been met well enough that this 